In [0]:
%run ../utils/adls_auth

In [0]:
%run ../utils/control_table

In [0]:
%run ../utils/dq_helpers

In [0]:
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col, explode, arrays_zip, to_date, to_timestamp, current_timestamp, lit, array, arrays_overlap
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, ArrayType, DoubleType, StringType


In [0]:
BRONZE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/weather_raw"
SCHEMA_LOCATION = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_schemas/weather_autoloader"
CHECKPOINT_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/_checkpoints/weather_silver"
SILVER_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/weather_silver"
QUARANTINE_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/weather_quarantine"

silver_table_exists = DeltaTable.isDeltaTable(spark, SILVER_PATH)


In [0]:


def process_batch(batch_df, batch_id):
    global silver_table_exists
    run_batch_id = f"{batch_id}_{uuid.uuid4()}"

    zipped_df = batch_df.select(
        explode(arrays_zip(
            col("hourly.time"), col("hourly.temperature_2m"), col("hourly.precipitation"),
            col("hourly.snowfall"), col("hourly.wind_speed_10m"), col("hourly.weathercode"),
        )).alias("r")
    ).select(
        to_timestamp(col("r.time")).alias("weather_datetime"),
        col("r.temperature_2m").alias("temperature_c"),
        col("r.precipitation").alias("precipitation_mm"),
        col("r.snowfall").alias("snowfall_cm"),
        col("r.wind_speed_10m").alias("wind_speed_kmh"),
        col("r.weathercode").alias("weather_code"),
    )

    before_dedupe = zipped_df.count()
    if before_dedupe == 0:
        print(f"[batch {batch_id}] empty batch, skipping.")
        return
    zipped_df = zipped_df.dropDuplicates(["weather_datetime"])
    duplicates_removed = before_dedupe - zipped_df.count()
    log_dq_result(
        spark, run_batch_id, "silver", "weather_silver", "duplicate_weather_timestamp",
        severity="CRITICAL", rows_checked=before_dedupe, rows_failed=duplicates_removed,
        action_taken="Duplicates dropped within micro-batch via dropDuplicates.",
    )

    checks = {
        "null_datetime": col("weather_datetime").isNull(),
        "temperature_out_of_range": (col("temperature_c") < -50) | (col("temperature_c") > 50),
        "negative_precipitation": col("precipitation_mm") < 0,
        "negative_snowfall": col("snowfall_cm") < 0,
    }
    flagged_df = flag_row_level_checks(zipped_df, checks)
    critical_array = array(*[lit("null_datetime")])
    flagged_df = flagged_df.withColumn("_has_critical_failure", arrays_overlap(col("_dq_failures"), critical_array))

    quarantine_df = flagged_df.filter(col("_has_critical_failure")).withColumn("_quarantined_at", current_timestamp())
    clean_df = flagged_df.filter(~col("_has_critical_failure")).withColumn("weather_date", to_date(col("weather_datetime")))


        # --- Monitor Auto Loader's rescued-data safety net ---
    if "_rescued_data" in clean_df.columns:
        rescued_count = clean_df.filter(col("_rescued_data").isNotNull()).count()
        if rescued_count > 0:
            log_dq_result(
                spark, run_batch_id, "silver", "weather_silver", "rescued_data_present",
                severity="WARNING", rows_checked=clean_df.count(), rows_failed=rescued_count,
                action_taken="Rows contain unrecognized fields in _rescued_data — investigate source schema before next run.",
            )
            print(f"WARNING: {rescued_count} rows have unrecognized fields in _rescued_data.")




    critical_failed = quarantine_df.count()
    log_dq_result(
        spark, run_batch_id, "silver", "weather_silver", "critical_row_checks",
        severity="CRITICAL", rows_checked=before_dedupe, rows_failed=critical_failed,
        action_taken="Rows with null datetime routed to weather_quarantine.",
    )

    if not silver_table_exists:
        clean_df.write.format("delta").mode("overwrite").partitionBy("weather_date").save(SILVER_PATH)
        silver_table_exists = True
    else:
        silver_table = DeltaTable.forPath(spark, SILVER_PATH)

        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
        (silver_table.alias("target")
            .merge(clean_df.alias("source"), "target.weather_datetime = source.weather_datetime")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")  

    quarantine_df.write.format("delta").mode("append").save(QUARANTINE_PATH)
    log_ingestion_event(
        spark, run_batch_id, "weather_silver_transform", f"batch_{batch_id}",
        status="SUCCESS", rows_written=clean_df.count(), started_at=datetime.utcnow(),
    )
    print(f"[batch {batch_id}] merged {clean_df.count()} hourly rows, quarantined {critical_failed}.")


In [0]:
weather_schema_hint_ddl = "hourly STRUCT<`time`: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, wind_speed_10m: ARRAY<DOUBLE>, weathercode: ARRAY<DOUBLE>>"

autoloader_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaHints", weather_schema_hint_ddl)
    .load(BRONZE_PATH)
)
query = (
    autoloader_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

try:
    query.awaitTermination()
    print("Weather Silver run complete (Auto Loader caught up, query terminated).")
except Exception as e:
    error_text = str(e)
    if "SCHEMA" in error_text.upper() or "SchemaColumnTypeException" in error_text or "DELTA_FAILED_TO_MERGE_FIELDS" in error_text.upper():
        log_dq_result(
            spark, str(uuid.uuid4()), "silver", "trips_silver", "breaking_schema_change_detected",
            severity="CRITICAL", rows_checked=0, rows_failed=0,
            action_taken=f"Pipeline halted — non-additive schema change detected: {error_text[:500]}",
        )
        print(f"CRITICAL: breaking schema change detected, pipeline halted.\n{error_text}")
    raise